# 06 · The integrated perception loop on the same window

`perception.run_perception` is what `run_analysis.py` runs. It does 02–05 in **one loop with feedback** the
notebooks above do not have: the detector ROI comes from the *previous* frame's pose, the clicked kits seed the
tracker, the PTZ camera motion drives the ball filter, off-pitch people are typed per track, bad camera frames are
masked. Comparing it with the stage-by-stage tables catches integration bugs (a stage that works alone but not in
the loop) and gives the KPIs to log per experiment.

In [ ]:
import sys, os, time, json, warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "football_eval").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 3); pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 60)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
from football_eval import nbtools as nbt

P = nbt.load_params()            # notebooks/pipeline/params.yaml (or $FVE_NB_PARAMS)
ctx = nbt.build_context(P)       # config <- match setup, restricted to the window
cfg = ctx.cfg

In [ ]:
from football_eval.perception import run_perception, save_perception, load_perception
ctx2 = nbt.build_context(P, verbose=False)               # fresh config: run_perception mutates it
with nbt.Timer() as tm:
    res = run_perception(ctx2.video, ctx2.cfg, progress=True, live_dir=None, team_names=list(ctx2.cfg.teams.team_names))
save_perception(res, str(ctx.out / "perception"))
m = res["meta"]
print(f"\n{m['n_frames_analysed']} frames in {tm.s:.0f}s -> {m['speed']['fps']} fps | detector {m['speed']['detector']} | GPU {m['speed']['gpu']} | "
      f"tiles/frame {m['speed']['tiles_per_frame']}")

## 1 · Integrated vs stage by stage

In [ ]:
S = {k: nbt.load_stage(ctx, k, required=False) for k in ("detections", "homography", "tracks", "ball")}
T, B, Fr = res["tracks"], res["ball"], res["frames"]
def kpis_stage():
    d, h, tr, b = S["detections"], S["homography"], S["tracks"], S["ball"]
    return {"people / frame (median)": d[d.cls != "ball"].groupby("frame").size().median() if d is not None else np.nan,
            "frames with pose": h.has_h.mean() if h is not None else np.nan,
            "confirmed tracks": tr[tr.track_id >= 0].track_id.nunique() if tr is not None else np.nan,
            "team-labelled rows": (tr.team >= 0).mean() if tr is not None else np.nan,
            "ball visible": b.visible.mean() if b is not None else np.nan,
            "ball usable (with gaps filled)": b.valid.mean() if b is not None else np.nan}
k_int = {"people / frame (median)": Fr.n_people.median(), "frames with pose": Fr.has_h.mean(),
         "confirmed tracks": T[T.track_id >= 0].track_id.nunique(), "team-labelled rows": (T.team >= 0).mean(),
         "ball visible": B.visible.mean(), "ball usable (with gaps filled)": B.valid.mean()}
cmp_ = pd.DataFrame({"stage notebooks (02-05)": kpis_stage(), "integrated loop": k_int})
cmp_["diff"] = cmp_.iloc[:, 1] - cmp_.iloc[:, 0]
cmp_.round(3)

Expected differences: more people / frame in the loop when the pitch ROI (from the pose) is wider than the grass ROI; fewer confirmed tracks when the kit seeds keep IDs; a different ball share because 05 used the grass ROI and no seeds. A large gap on *frames with pose* or *team-labelled rows* is worth a look.

## 2 · Look at a frame of the loop's output

In [ ]:
from football_eval.pitch import Pitch, draw_pitch
f, t, img = nbt.probe_frame(ctx)
fr_ = T.frame.iloc[(T.frame - f).abs().argmin()]
g = T[T.frame == fr_]; names = list(ctx2.cfg.teams.team_names)
col = [nbt.COLORS[int(r.team)] if r.team >= 0 else (160, 160, 160) for r in g.itertuples()]
lab = [f"{r.track_id} {r.cls[:3]}" for r in g.itertuples()]
vis = nbt.draw_boxes(nbt.read_frame_idx(ctx.video, int(fr_)), g[["x0", "y0", "x1", "y1"]].to_numpy(), lab, col)
bb = res["ball_raw"]; bb = bb[bb.frame == fr_]
for r in bb.itertuples():
    cv2.circle(vis, (int(r.u), int(r.v)), 18, nbt.COLORS["ball"], 3)
nbt.show(vis, f"integrated loop, frame {fr_}");

## 3 · Log the experiment
One row per run in `outputs/_pipeline_nb/benchmarks.csv` (commit, detector, thresholds, KPIs). Give it a label when you change something (`label="yolo11s finetuned, conf 0.25"`).

In [ ]:
kp = {k.split(" (")[0].replace(" / ", "_per_").replace(" ", "_"): round(float(v), 3) for k, v in k_int.items()}
kp.update(fps=m["speed"]["fps"], runtime=m["speed"]["detector"], gpu=m["speed"]["gpu"])
bench = nbt.log_benchmark(ctx2, kp, label="baseline")
bench.tail(8)